In [ ]:
%pip install pyodbc
%pip install sqlalchemy

In [ ]:
import pyodbc   
from sqlalchemy import create_engine
import pandas as pd

In [ ]:
server_name = 'YOUR_SQL_SERVER_INSTANCE'
database_name = 'YOUR_DATABASE_NAME'


# username = ''
# password = ''


# Connection string:
# Connection_string = (
#                        f"DRIVER = {{ODBC Driver 17 for SQL Server}};"
#                        f"SERVER = {server_name}; DATABASE = {database_name};"
#                         "Trusted_Connection = yes;"
#                     )


# depending upon what rdbms system you are using, the below change.
connection_string = f"mssql+pyodbc://@{server_name}/{database_name}?driver=ODBC+Driver+17+for+SQL+Server"

In [ ]:
connection_string

In [ ]:
connection2 = create_engine(connection_string)
connection2

In [ ]:
## just to check if the connection is successful or not

query = "SELECT TOP 5 * FROM products_p"

df = pd.read_sql(query, connection2)

df.head()

In [ ]:
#--------------------- Importing the cleaned tables

In [ ]:
query = "SELECT * FROM products_p"

products = pd.read_sql(query, connection2)

products.head()

In [ ]:
products.shape
products.info()
products.isnull().sum()

In [ ]:
query = """
SELECT TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME
"""

pd.read_sql(query, connection2)

In [ ]:
tables = [
    'customers',
    'orders_o',
    'order_items',
    'order_payments_op',
    'order_reviews_or',
    'sellers',
    'geolocations_gl',
    'product_category_name_translation', 'order_payments'
]

dataframes = {}

for i in tables:
    dataframes[i] = pd.read_sql(
        f"SELECT * FROM [{i}]",
        connection2
    )
    print(i, dataframes[i].shape)

In [ ]:
customers = dataframes['customers']
orders = dataframes['orders_o']
order_items = dataframes['order_items']
order_payments = dataframes['order_payments_op']
order_reviews = dataframes['order_reviews_or']
sellers = dataframes['sellers']
geolocation = dataframes['geolocations_gl']
product_category_name_translation = dataframes['product_category_name_translation']
order_payment_raw = dataframes['order_payments']

In [ ]:
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
table = {'customers':customers, 'orders':orders, 'order_items':order_items,'order_payments':order_payments,
         'order_reviews':order_reviews, 'sellers':sellers, 'geolocation':geolocation, 'products':products, 
         'product_category_name_translation':product_category_name_translation}

In [ ]:
# Column-wise missing values and percentage

for i, j in table.items():
    missing = j.isnull().sum()
    missing_pct = (missing / len(j) * 100)

    missing_summary = pd.DataFrame({'missing_values': missing,'missing_percentage': missing_pct})

    missing_summary = missing_summary[missing_summary['missing_values'] > 0]

    if not missing_summary.empty:
        print(f'\n--- {j} ---')
        print(missing_summary)

In [ ]:
print(orders.isna().sum(),'\n',orders.isna().sum()/len(orders)*100)

In [ ]:
print(order_reviews.isna().sum(),'\n',order_reviews.isna().sum()/len(orders)*100)

In [ ]:
print(geolocation.isna().sum(),'\n',geolocation.isna().sum()/len(orders)*100)

In [ ]:
print(products.isna().sum(),'\n',products.isna().sum()/len(orders)*100)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,classification_report,confusion_matrix)
from sklearn.model_selection import train_test_split

In [ ]:
# 1. Table Merging at Order Level
seller_agg = order_items.groupby('order_id', as_index=False).agg(unique_sellers=('seller_id', 'nunique'),total_items=('order_item_id', 'count'),)

review_ord_seller = (order_reviews.drop_duplicates('order_id').merge(orders, on='order_id', how='inner').merge(seller_agg, on='order_id', how='left'))


In [ ]:
# 2. Date Conversions & Operational Features
date_cols = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
             'order_delivered_customer_date','order_estimated_delivery_date','review_creation_date',]

for i in date_cols:
  review_ord_seller[i] = pd.to_datetime(review_ord_seller[i])

review_ord_seller['is_delivered'] = (review_ord_seller['order_status'] == 'delivered').astype(int)
review_ord_seller['delivery_days'] = (review_ord_seller['order_delivered_customer_date'] - review_ord_seller['order_purchase_timestamp']).dt.days.fillna(0)
review_ord_seller['delay_days'] = (review_ord_seller['order_delivered_customer_date'] - review_ord_seller['order_estimated_delivery_date']).dt.days.fillna(0)
review_ord_seller['transit_days'] = (review_ord_seller['order_delivered_customer_date'] - review_ord_seller['order_delivered_carrier_date']).dt.days.fillna(0)
review_ord_seller['carrier_handoff_days'] = (review_ord_seller['order_delivered_carrier_date'] - review_ord_seller['order_approved_at']).dt.days.fillna(0)
review_ord_seller['survey_before_delivery'] = (review_ord_seller['order_delivered_customer_date'] > review_ord_seller['review_creation_date']).astype(int)
review_ord_seller['is_late'] = (review_ord_seller['delay_days'] > 0).astype(int)
review_ord_seller['unique_sellers'] = review_ord_seller['unique_sellers'].fillna(1)
review_ord_seller['total_items'] = review_ord_seller['total_items'].fillna(1)


In [ ]:
# Target: 3 Exact Classes
y = np.where(review_ord_seller['review_score'] <= 2,'Negative [1-2]',
    np.where(review_ord_seller['review_score'] == 3,'Neutral [3]','Positive [4-5]',),)
review_ord_seller['actual_review'] = y

features = ['delay_days','is_late','delivery_days','transit_days','carrier_handoff_days',
            'survey_before_delivery','unique_sellers','total_items',]

X = review_ord_seller[features]

### Feature correlation heatmap
Pearson correlation across the model input features. The target class is excluded.


In [ ]:
feature_corr = X.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(feature_corr,annot=True,fmt=".2f",cmap="RdBu_r",vmin=-1,vmax=1,center=0,square=True,linewidths=0.5)
plt.title("Correlation Between Model Features")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
# 4. Multinomial Logistic Regression Model Training
model_log = LogisticRegression(multi_class='multinomial',class_weight='balanced',max_iter=1000,random_state=42,)
model_log.fit(X_train, y_train)


In [ ]:
# 5. Model Evaluation
y_pred = model_log.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'Accuracy: {acc * 100:.2f}% ')


In [ ]:
print('Classification Report:\n', classification_report(y_test, y_pred, digits=4))

In [ ]:
# confusion matrix

labels_order = ['Negative [1-2]', 'Neutral [3]', 'Positive [4-5]']
cm = confusion_matrix(y_test, y_pred, labels=labels_order)
cm

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(cm,annot=True,fmt='d',xticklabels=labels_order,yticklabels=labels_order)

plt.xlabel('Predicted Class')
plt.ylabel('Actual Class')

In [ ]:
review_ord_seller['predicted_review'] = model_log.predict(X)

# Negative [1-2] (Dissatisfaction Risk)
negatives = list(model_log.classes_).index('Negative [1-2]')
review_ord_seller['risk_probability'] = model_log.predict_proba(X)[:, negatives].round(4)

comparison_cols = ['order_id','review_score','actual_review','predicted_review',
                   'risk_probability','delay_days','delivery_days','survey_before_delivery']

review_ord_seller[comparison_cols]